In [1]:
import os
import numpy as np
import copy as cp
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
import math
import collections
import pickle
import string

In [2]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.titlesize'] = 8    # Size of the plot title
plt.rcParams['axes.labelsize'] = 8    # Size of x and y labels
plt.rcParams['xtick.labelsize'] = 7   # Size of x-tick labels
plt.rcParams['ytick.labelsize'] = 7   # Size of y-tick labels
plt.rcParams['legend.fontsize'] = 7
plt.rcParams['font.size'] = 8

plt.rcParams.update({
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'xtick.minor.size': 2,
    'ytick.minor.size': 2,
    'xtick.major.pad': 2,
    'ytick.major.pad': 2,
    'axes.linewidth': 0.8,
})

cm_to_inch = 2.54

In [3]:
inputdir  = '../results/CMEs/2026-09-27/'
outputdir = inputdir
save      = True

In [4]:
studies = ['GB1', 'ParD3-ParE3', 'ParD3-ParE2']

study_dictionary = collections.defaultdict (dict)
study_dictionary['GB1']['M'] = 4
study_dictionary['ParD3-ParE3']['M'] = 3
study_dictionary['ParD3-ParE2']['M'] = 3

#study = 'GB1'

my_colors  = ['gold','cornflowerblue','salmon','dimgray','lightgray']
scenarios  = ['beneficial','deleterious','adaptive','conditional','random']
color_dict = dict (zip (scenarios, my_colors))
ntypes = len (scenarios)

nsims  = 100

In [5]:
for study in studies :
    study_dictionary[study]['R2']     = pickle.load (open( os.path.join (inputdir, study, 'R2_' + study + '.pkl'), 'rb'))
    study_dictionary[study]['nMax']   = pickle.load (open( os.path.join (inputdir, study, 'nmax_' + study + '.pkl'), 'rb'))
    study_dictionary[study]['nPaths'] = np.loadtxt ( os.path.join (inputdir, study, 'npaths_' + study + '.txt') )
    study_dictionary[study]['Yd']     = pickle.load (open( os.path.join (inputdir, study, 'Yd_' + study + '.pkl'), 'rb'))
    study_dictionary[study]['Ad']     = pickle.load (open( os.path.join (inputdir, study, 'Ad_' + study + '.pkl'), 'rb'))
    study_dictionary[study]['phenos'] = np.loadtxt (os.path.join (inputdir, study, 'phenos_' + study + '.txt') )
    study_dictionary[study]['F1s']    = np.loadtxt (os.path.join (inputdir, study, 'f1s_' + study + '.txt'))
    study_dictionary[study]['R2_all'] = np.loadtxt (os.path.join (inputdir, study, 'ols_reg_' + study + '.txt'))

study_dictionary['ParD3-ParE2']['wt'] = 953
study_dictionary['ParD3-ParE3']['wt'] = 953
study_dictionary['GB1']['wt'] = 0

In [6]:
# plot R2 results
plt.rcParams['figure.figsize'] = (17.8/cm_to_inch,1.4*3)
fig, axs = plt.subplots (3, 5, constrained_layout=True)

# A. R2
ct = 0
for study in studies :

    ax = axs[ct, 0]

    R2 = cp.deepcopy (study_dictionary[study]['R2'])
    R2_all = cp.deepcopy (study_dictionary[study]['R2_all'])

    M  = study_dictionary[study]['M']
    
    # x labels
    xs = np.arange (0,M+1,1)
    for i in range (ntypes) :
        ax.scatter (xs+.05*i, np.nanmean (R2[i,:,:], axis=0), color=color_dict[scenarios[i]], edgecolor='black', lw=.5)
        ax.errorbar(xs+.05*i, np.mean (R2[i,:,:], axis=0), yerr=1.96 * np.sqrt (np.var (R2[i,:,:], axis=0)/nsims),
                    fmt='', color=color_dict[scenarios[i]], linestyle='-', linewidth=.5, ecolor=color_dict[scenarios[i]], capsize=4)
    
    # hoc
    hoc = np.array ([math.comb (M, i) / (2**M-1) for i in range (M+1)])
    hoc[0] = 0
    ax.plot (xs, np.cumsum (hoc) / np.sum (hoc), linestyle='--', lw=.5, color='black')
    ax.plot (xs, np.append (R2_all, [1]), color='black', marker='x', linewidth=.5, label=r'global')
    
    R2_add = np.ones (M+1)
    R2_add[0] = 0
    ax.plot (xs, R2_add, linestyle='-.', lw=.5, color='black')
    
    ax.set_xticks (xs, xs)
    ax.set_ylabel (r'$R^2_{n}$', labelpad=-.75)

    if ct == 0 :
        ax.legend (frameon=False)
    

    if ct == len (studies) - 1 :
        ax.set_xlabel (r'model order, $n$')

    ct += 1


## B + C
ct = 0
for study in studies :

    # B
    ax = axs[ct,1]
    nMax = cp.deepcopy (study_dictionary[study]['nMax'])
    W    = np.sum (nMax, axis=2)
    vplot = ax.violinplot (np.transpose (W), showmedians=True)
    for i, body in enumerate(vplot['bodies']):
        body.set_facecolor(color_dict[scenarios[i]])
        body.set_edgecolor('black')    # Border color
        body.set_alpha(.8)
    
    for el in ['cbars','cmedians','cmaxes','cmins'] :
        vplot[el].set_color ('black')
        vplot[el].set_linewidth (.5)
    
    vplot['cmedians'].set_linewidth (1)
    
    ax.axhline (1, linestyle='-.', lw=.5, color='black')
    ax.axhline (2**M / (M+1), linestyle='--', lw=.5, color='black')
    
    ax.set_ylabel (r'$W$', labelpad=-.75)

    if ct == len (studies) - 1 :
        ax.set_xticks (np.arange (1,ntypes+1,1), [x[:4] for x in scenarios], rotation=75)
    else :
        ax.set_xticks (np.arange (1,ntypes+1,1), ['']*ntypes, rotation=75)
        ax.set_xticklabels([])

    # Jittered points
    for i, y in enumerate(W, start=1):
        x = np.random.normal(i, 0.1, size=len(y))
        yjit = np.random.normal(0, 0.1, size=len(y))
        ax.scatter(x, y+yjit, color=my_colors[i-1], edgecolor='black', lw=.2, alpha=.6, s=2, zorder=1, label=scenarios[i-1][:4])

    # C
    M  = study_dictionary[study]['M']
    xs = np.arange (0,M+1,1)
    ax = axs[ct,2]    
    for i in range (ntypes) :
        ax.scatter (xs+.05*i, np.nanmean (nMax[i,:,:], axis=0),
                         label=scenarios[i], color=color_dict[scenarios[i]], edgecolor='black', lw=.5)
        ax.errorbar(xs+.05*i, np.mean (nMax[i,:,:], axis=0), yerr=1.96 * np.sqrt (np.var (nMax[i,:,:], axis=0)/nsims),
                   fmt='', color=color_dict[scenarios[i]], linestyle='-', linewidth=.5, ecolor=color_dict[scenarios[i]], capsize=4)
    
    # hoc
    hoc = np.array ([math.comb (M, i) / (M+1) for i in range (M+1)])
    ax.plot (xs, hoc, linestyle='--', lw=.5, color='black', label='HoC')
    
    add = np.array ([math.comb (M, x) for x in range (M+1)])
    
    ax.plot (xs, add*2**(-M), linestyle='-.', lw=.5, color='black')

    ax.set_xticks (xs, xs)
    ax.set_ylabel (r'$W_m$', labelpad=-.75)

    if ct == len (studies) - 1 :
        ax.set_xlabel (r'distance, $m$')

    ct += 1

# D
ct = 0
for study in studies :
    ax = axs[ct, 3]

    nPaths = cp.deepcopy (study_dictionary[study]['nPaths'])
    xs = np.arange (0,ntypes,1)
    
    flierprops = dict(marker='o', markerfacecolor='yellow',
                      markeredgecolor='black', markersize=0)
    
    box = ax.violinplot ( np.transpose (nPaths), showmedians=True )
    
    for i, body in enumerate(box['bodies']):
        body.set_facecolor(color_dict[scenarios[i]])
        body.set_edgecolor('black')    # Border color
        body.set_alpha(.8)
    
    for el in ['cbars','cmedians','cmaxes','cmins'] :
        box[el].set_color ('black')
        box[el].set_linewidth (.5)
    
    box['cmedians'].set_linewidth (1)
    
    # Jittered points
    for i, y in enumerate(nPaths, start=1):
        x = np.random.normal(i, 0.1, size=len(y))
        yjit = np.random.normal(0, 0.1, size=len(y))
        ax.scatter(x, y+yjit, color=my_colors[i-1], edgecolor='black', lw=.2, alpha=.6, s=2, zorder=1, label=scenarios[i-1][:4])
    
    ax.axhline (1, color='black', linestyle='--', lw=.5, label='HoC')
    ax.axhline (math.factorial (study_dictionary[study]['M']), color='black', linestyle='-.', lw=.5, label='add')

    ax.set_ylabel (r'$P$', labelpad=-.75)

    if ct == len (studies) - 1 :
        ax.set_xticks (xs+1, [x[:4] for x in scenarios], rotation=75)
    else :
        ax.set_xticks (np.arange (1,ntypes+1,1), ['']*ntypes, rotation=75)
        ax.set_xticklabels([])
    
    ct += 1

# E
ct = 0
for study in studies :
    ax = axs[ct, 4]

    Yd = cp.deepcopy (study_dictionary[study]['Yd'])
    Ad = cp.deepcopy (study_dictionary[study]['Ad'])
    mu = np.mean (study_dictionary[study]['phenos'])

    M = study_dictionary[study]['M']
    xs = np.arange (0, M+1, 1)
    for i in range (ntypes) :
        ax.plot (xs, np.mean (Yd[i,:,:], axis=0), color=color_dict[scenarios[i]], lw=.5, label=scenarios[i][:4])
        ax.errorbar (xs, np.mean (Yd[i,:,:], axis=0), yerr=1.96*np.sqrt (np.var (Yd[i,:,:], axis=0)/nsims),
                      color=color_dict[scenarios[i]], lw=.5, linestyle=None, capsize=4)
        ax.scatter (xs, np.mean (Yd[i,:,:], axis=0), edgecolor='black', lw=.5,
                      color=color_dict[scenarios[i]], zorder=10)
        ax.errorbar (xs, np.mean (Ad[i,:,:], axis=0) + mu, yerr=1.96*np.sqrt (np.var (Ad[i,:,:], axis=0)/nsims),
                      color=color_dict[scenarios[i]], lw=.5, capsize=4, linestyle='dotted') #, label=r'$\hat Y_\text{add}$')
        ax.scatter (xs, np.mean (Ad[i,:,:], axis=0) + mu, marker='v', edgecolor='black', lw=.5, color=color_dict[scenarios[i]], zorder=10)

    ax.set_ylabel (r'fitness, $F$', labelpad=-.75)
    ax.set_xticks ( np.arange (0,M+1,1), np.arange (0,M+1,1))

    if ct == len (studies) - 1 :
        ax.set_xlabel (r'distance, $m$')

    ct += 1


# Figure legends
# add vs observed legend
marker_circle = Line2D([0], [0], markerfacecolor='white',
                         markeredgecolor='black', markeredgewidth=.5, marker='o',
                         linestyle='-', linewidth=.5, color='black', label=r'$F$')

marker_triangle = Line2D([0], [0], markerfacecolor='white',
                         markeredgecolor='black', markeredgewidth=.5, marker='v',
                         linestyle='dotted', linewidth=.5, color='black', label=r'$\hat F_\text{add}$')

fig.legend(
    handles=[marker_circle, marker_triangle], 
    bbox_to_anchor=(.92,1.1),
    loc='outside upper center',
    frameon=False
)

# scenario legend
legend_elements = []
for scen in scenarios :
    legend_elements.append ( Line2D([0], [0], marker='o', color='w',
                             markerfacecolor=color_dict[scen], markeredgecolor='black',
                                    markeredgewidth=.5, markersize=7, label=scen) )
legend_elements.append (Line2D([0], [0], color='white', lw=.5, label=''))
legend_elements.append (Line2D([0], [0], color='black', lw=.5, label='HoC', linestyle='dashed'))
legend_elements.append (Line2D([0], [0], color='black', lw=.5, label='additive', linestyle='-.'))

fig.legend (handles=legend_elements, loc='outside upper center',
            ncol=4, frameon=False, bbox_to_anchor=(.43,1.1),
            labelspacing=.5, columnspacing=.5, handletextpad=0.4)


for j in range (5) :
    axs[0,j].set_title (string.ascii_uppercase[j], pad=1, loc='left', fontweight='bold')


if save :
    plt.savefig (os.path.join (outputdir, 'statistical_all_' + str (nsims) + '.pdf'), bbox_inches='tight')
    plt.close ()
else :
    plt.show ()




In [7]:
plt.rcParams['figure.figsize'] = (11/cm_to_inch,1.4)

fig, axs = plt.subplots (1, len (studies), constrained_layout=True)

ct = 0
for study in studies :
    axs[ct].hist (study_dictionary[study]['phenos'], color='lightgray', edgecolor='black', lw=.5)
    axs[ct].axvline (study_dictionary[study]['phenos'][study_dictionary[study]['wt']],
                     linewidth=.5, color='black', linestyle='--', label=r'$F(\mathbf{x}_{wt})$')
    axs[ct].set_xlabel (r'fitness, $F$')
    axs[ct].set_title (string.ascii_uppercase[ct], pad=1, loc='left', fontweight='bold')
    
    if ct == (len (studies)-1) :
        axs[ct].legend (frameon=False, handlelength=1)
    ct += 1

axs[0].set_ylabel ('Count')

if save :
    plt.savefig (os.path.join (outputdir, 'data_F_histograms.pdf'), bbox_inches='tight')
    plt.close ()
else :
    plt.show ()
